# 도구 호출(Tool Calling) 기초

LLM은 글을 만들 뿐, 파일에 저장하거나 메일을 보내지는 못합니다.
**도구 호출**은 이 한계를 넘는 방법입니다. LLM이 "이 함수를 이런 값으로 불러 주세요"라고 요청하면,
우리 코드가 그 함수를 대신 실행하고 결과를 다시 알려 줍니다.

이 노트북에서는 **상담 도중 사용자의 이름·전화번호·주소를 받아 파일에 저장하는** 챗봇을 만듭니다.

## 전체 흐름

```
① 파이썬 함수를 만든다              record_personal_info_tool()
② LLM에게 그 함수를 설명한다         JSON 스키마
③ LLM이 "이 함수를 부르세요"라고 답한다   finish_reason == "tool_calls"
④ 우리 코드가 함수를 실행한다
⑤ 결과를 LLM에게 돌려주고 최종 답변을 받는다
```

**중요**: LLM이 함수를 직접 실행하는 것이 아닙니다. **실행은 항상 우리 코드가 합니다.**
LLM은 "무엇을, 어떤 값으로 부를지"만 정합니다.

> ⚠️ 실습에는 **가짜 개인정보**를 쓰세요. 실제 이름·전화번호·주소를 입력하면 `personal_info.txt`에 그대로 저장됩니다.

## Step 1. 준비

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import gradio as gr
#from IPython.display import Markdown, display

load_dotenv(override=True)
openai = OpenAI()

C:\aiwins\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2. 시스템 프롬프트 — 도구를 언제 쓸지 알려 주기

도구를 등록하는 것만으로는 부족합니다. **언제 부를지**를 프롬프트로 알려 줘야 합니다.

아래 프롬프트에는 규칙이 두 개 들어 있습니다.

1. 이름·전화번호·주소 **세 가지를 모두 받으면** 도구를 호출한다.
2. 다 모이기 전에는 **호출하지 말고** 빠진 항목을 물어본다.

두 번째 규칙이 없으면 모델이 빈 값으로 도구를 부르거나, 값을 지어내기도 합니다.

In [2]:
system_prompt = """
당신은 대화 전문가이며 대화하는 사람의 고민을 들어주고 제안을 할 수 있습니다.

대화 중에 상담 기록을 남기기 위해 사용자의 이름, 전화번호, 주소를 물어보세요.
세 가지를 모두 받으면 record_personal_info_tool 도구를 호출해서 기록하세요.
세 가지가 다 모이기 전에는 도구를 호출하지 말고, 빠진 항목을 물어보세요.
"""

## Step 3. 먼저 도구 없이 대화해 보기

비교를 위해 도구를 주지 않고 한 번 호출해 봅니다.
프롬프트에 "기록하라"고 적혀 있어도, **도구가 없으면 모델은 말만 할 뿐 아무것도 저장하지 못합니다.**

In [3]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "나의 고민을 해결할 수 있는 방법을 알려 주세요"},
]

In [4]:
response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
print(response.choices[0].message.content)

물론입니다. 어떤 고민이든 편하게 말씀해 주세요.  
제가 함께 정리하고 가능한 해결 방법을 찾아드릴게요.

상황을 더 잘 이해하려면 아래 정보도 알려주시면 도움이 됩니다:
- 지금 가장 힘든 점이 무엇인지
- 언제부터 이런 고민이 있었는지
- 원하시는 방향이 있는지

그리고 상담 기록을 남기기 위해 아래 3가지 정보도 필요합니다:
1. 이름  
2. 전화번호  
3. 주소  

세 가지를 모두 알려주시면 기록해 드릴게요.


## Step 4. 도구로 쓸 파이썬 함수 만들기

특별한 것은 없습니다. **그냥 평범한 파이썬 함수**입니다.

- 파일에 한 줄씩 덧붙여 저장합니다(`"a"` 모드).
- `print`를 넣어 두면 도구가 실제로 불렸는지 눈으로 확인할 수 있습니다.
- 마지막에 **문자열을 돌려줍니다.** 이 값이 나중에 LLM에게 "작업 결과"로 전달됩니다.

다음 셀에서 함수를 직접 호출해 정상 동작을 먼저 확인합니다. LLM과 연결하기 전에
**함수만 따로 시험해 보는 것**이 문제를 줄이는 방법입니다.

In [5]:
def record_personal_info_tool(name, phone, address):
    print(f"Tool called to record: {name} / {phone} / {address}")
    with open("personal_info.txt", "a", encoding="utf-8") as f:
        f.write(f"{name}\t{phone}\t{address}\n")
    return "personal info is recorded"

In [6]:
record_personal_info_tool("홍길동", "(051) 777-9999", "부산시 해운대구")

Tool called to record: 홍길동 / (051) 777-9999 / 부산시 해운대구


'personal info is recorded'

## Step 5. LLM에게 도구를 설명하기 (JSON 스키마)

### 방법 A: 스키마를 직접 만들기

> Step 8에서 같은 도구를 **자동으로** 만드는 방법(방법 B)과 비교합니다. 먼저 손으로 만들어 봅니다.

LLM은 우리 파이썬 코드를 볼 수 없습니다. 그래서 **함수 설명서**를 JSON으로 만들어 넘겨야 합니다.

| 항목 | 역할 |
|---|---|
| `name` | 함수 이름. 실제 파이썬 함수 이름과 같게 맞추면 관리하기 쉽습니다 |
| `description` | **이 도구를 언제 쓰는지.** 모델이 호출 여부를 판단하는 근거입니다 |
| `properties` | 각 인자의 이름·타입·설명 |
| `required` | 반드시 있어야 하는 인자 |

`description`을 성의 없이 적으면 모델이 도구를 엉뚱한 때에 부르거나 아예 부르지 않습니다.
**설명서가 곧 사용 설명**이라고 생각하세요.

In [7]:
record_personal_info_tool_json = {
    "name": "record_personal_info_tool",
    "description": "Use this tool to record the name, phone number and address that a user provided",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "The name of this user"},
            "phone": {"type": "string", "description": "The phone number of this user"},
            "address": {"type": "string", "description": "The address of this user"}
        },
        "required": ["name", "phone", "address"],
        "additionalProperties": False
    }
}

도구는 여러 개를 등록할 수 있으므로 **리스트**로 감쌉니다.
각 항목은 `{"type": "function", "function": 설명서}` 형태입니다.

In [8]:
tools = [{"type": "function", "function": record_personal_info_tool_json}]

In [9]:
tools

[{'type': 'function',
  'function': {'name': 'record_personal_info_tool',
   'description': 'Use this tool to record the name, phone number and address that a user provided',
   'parameters': {'type': 'object',
    'properties': {'name': {'type': 'string',
      'description': 'The name of this user'},
     'phone': {'type': 'string',
      'description': 'The phone number of this user'},
     'address': {'type': 'string', 'description': 'The address of this user'}},
    'required': ['name', 'phone', 'address'],
    'additionalProperties': False}}}]

## Step 6. 도구 호출 처리하기 (이 노트북의 핵심)

여기서 LLM을 **두 번** 호출합니다. 도구 호출은 한 번의 대화로 끝나지 않기 때문입니다.

```
1차 호출  → 모델이 "record_personal_info_tool을 이 값으로 불러 주세요"라고 답함
           (finish_reason == "tool_calls")
   ↓
우리 코드가 함수를 실행
   ↓
2차 호출  → 실행 결과를 붙여서 다시 물어봄 → 모델이 사용자에게 할 최종 답변을 만듦
```

코드를 한 줄씩 보면 이렇습니다.

| 코드 | 하는 일 |
|---|---|
| `tools=tools` | 이번 요청에 쓸 수 있는 도구 목록을 함께 보냄 |
| `finish_reason == "tool_calls"` | 모델이 답변 대신 **도구 호출을 요청**했는지 확인 |
| `messages.append(assistant_message)` | 모델의 호출 요청을 대화 기록에 남김 (**빠뜨리면 오류**) |
| `json.loads(tool_call.function.arguments)` | 모델이 보낸 인자는 **JSON 문자열**이라 파싱이 필요 |
| `{"role": "tool", ...}` | 실행 결과를 `tool` 역할로 추가. `tool_call_id`로 어느 호출의 결과인지 연결 |

`tool_calls`가 아니면 보통의 답변이므로 그대로 돌려줍니다.

In [10]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        for tool_call in assistant_message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            result = record_personal_info_tool(
                args.get("name"), args.get("phone"), args.get("address")
            )
            messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})

        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    return response.choices[0].message.content

## Step 7. 실행해 보기

Gradio 챗봇으로 실제 대화를 해 봅니다. 이렇게 진행해 보세요.

1. "요즘 고민이 있어요" 처럼 아무 말이나 겁니다.
2. 모델이 이름·전화번호·주소를 물어봅니다.
3. **일부만** 알려 줍니다 → 도구를 부르지 않고 나머지를 물어보는지 확인하세요.
4. 나머지를 마저 알려 줍니다 → 노트북에 `Tool called to record: ...`가 찍히면 성공입니다.
5. `personal_info.txt` 파일을 열어 저장된 내용을 확인합니다.

> 다시 말씀드리지만 **가짜 정보**를 쓰세요.

In [11]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Step 8. 같은 도구를 `@function_tool`로 만들기 — 두 방식 비교

지금까지 한 방식에는 불편한 점이 있습니다. **함수와 JSON 스키마를 따로 관리**해야 합니다.

- 함수에 인자를 하나 추가하면 → 스키마도 고쳐야 합니다.
- 고치는 것을 잊으면 → 오류는 안 나고, 모델이 그 인자를 **영원히 채우지 않습니다.**

OpenAI **Agents SDK**의 `@function_tool` 데코레이터는 이 문제를 없앱니다.
함수의 **타입 힌트와 독스트링을 읽어 스키마를 자동으로 만들어** 줍니다.

> 설치: `pip install openai-agents`

In [12]:
from agents import Agent, Runner, function_tool

### 방법 B: 데코레이터로 만들기

함수 위에 `@function_tool`만 붙이면 끝입니다. 대신 두 가지를 지켜야 합니다.

1. **타입 힌트**(`name: str`)를 반드시 적습니다 → 스키마의 `type`이 됩니다.
2. **독스트링**에 설명과 `Args:`를 적습니다 → `description`이 됩니다.

In [13]:
@function_tool
def record_personal_info_tool_v2(name: str, phone: str, address: str) -> str:
    """
    사용자가 알려 준 이름, 전화번호, 주소를 파일에 기록한다.

    Args:
        name: 사용자 이름
        phone: 사용자 전화번호
        address: 사용자 주소
    """
    print(f"Tool called to record: {name} / {phone} / {address}")
    with open("personal_info.txt", "a", encoding="utf-8") as f:
        f.write(f"{name}\t{phone}\t{address}\n")
    return "personal info is recorded"

In [14]:
# 자동으로 만들어진 스키마를 확인해 봅니다.
print("이름:", record_personal_info_tool_v2.name)
print("설명:", record_personal_info_tool_v2.description)
print()
print(json.dumps(record_personal_info_tool_v2.params_json_schema, indent=2, ensure_ascii=False))

이름: record_personal_info_tool_v2
설명: 사용자가 알려 준 이름, 전화번호, 주소를 파일에 기록한다.

{
  "properties": {
    "name": {
      "description": "사용자 이름",
      "title": "Name",
      "type": "string"
    },
    "phone": {
      "description": "사용자 전화번호",
      "title": "Phone",
      "type": "string"
    },
    "address": {
      "description": "사용자 주소",
      "title": "Address",
      "type": "string"
    }
  },
  "required": [
    "name",
    "phone",
    "address"
  ],
  "title": "record_personal_info_tool_v2_args",
  "type": "object",
  "additionalProperties": false
}


### 무엇이 어디서 왔나

Step 5에서 손으로 쓴 것과 비교해 보세요. **내용이 같습니다.**

| 스키마 항목 | 방법 A (손으로) | 방법 B (데코레이터) |
|---|---|---|
| `name` | 직접 입력 | 함수 이름에서 자동 |
| `description` | 직접 입력 | **독스트링 첫 줄**에서 자동 |
| `type: "string"` | 직접 입력 | **타입 힌트 `: str`** 에서 자동 |
| 인자 설명 | 직접 입력 | 독스트링의 **`Args:`** 에서 자동 |
| `required` | 직접 입력 | 기본값 없는 인자에서 자동 |

실행해 보면 `title` 같은 항목이 몇 개 더 붙어 있는데, 모델이 읽는 데는 문제가 없습니다.

### 진짜 큰 차이는 스키마가 아닙니다

데코레이터의 이점은 "타이핑이 준다"는 정도지만, **Agents SDK를 쓰면 Step 6의 코드가 통째로 사라집니다.**

Step 6에서 우리가 손으로 한 일을 떠올려 보세요.

1. `finish_reason == "tool_calls"` 인지 확인하고
2. `arguments`를 `json.loads`로 풀고
3. 함수를 실행하고
4. 결과를 `{"role": "tool", ...}`로 붙이고
5. LLM을 **다시** 호출

이 다섯 단계를 **SDK가 알아서 합니다.** 우리는 도구를 등록하고 실행만 시키면 됩니다.

In [15]:
agent = Agent(
    name="상담 도우미",
    instructions=system_prompt,        # Step 2에서 만든 것을 그대로 사용
    model="gpt-5.4-mini",
    tools=[record_personal_info_tool_v2],
)

# 정보가 부족할 때 — 도구를 부르지 않고 물어봐야 한다
result = await Runner.run(agent, "요즘 고민이 많아요. 제 이름은 김철수입니다.")
print(result.final_output)

고민이 많으시군요. 편하게 말씀해 주세요.  
상담 기록을 위해 **전화번호**와 **주소**도 알려주실 수 있을까요?


In [16]:
# 세 가지를 모두 줄 때 — 도구를 부른다
result = await Runner.run(agent, "이름 김철수, 전화 010-1234-5678, 주소 서울시 강남구입니다.")
print(result.final_output)

Tool called to record: 김철수 / 010-1234-5678 / 서울시 강남구입니다.


기록했어요.


### Gradio에 붙이기

Step 6의 `chat()` 함수와 비교해 보세요. **한 줄로 줄었습니다.**
`Runner.run`은 비동기 함수이므로 `async def`로 정의하고 `await`를 씁니다.

In [17]:
def to_agent_input(history, message):
    """gradio 기록을 에이전트 입력으로 바꾼다. role/content만 남긴다.

    gradio는 options, metadata 같은 표시용 키를 함께 넘기는데,
    그대로 보내면 OpenAI API가 거부한다(Unknown parameter: 'input[0].options').
    """
    items = []
    for msg in history:
        role = msg.get("role")
        content = msg.get("content")
        if role in ("user", "assistant") and isinstance(content, str):
            items.append({"role": role, "content": content})
    items.append({"role": "user", "content": message})
    return items


async def chat_with_agent(message, history):
    result = await Runner.run(agent, to_agent_input(history, message))
    return result.final_output


gr.ChatInterface(chat_with_agent).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


### 두 방식 비교 정리

| | 방법 A: 직접 만들기 | 방법 B: Agents SDK |
|---|---|---|
| 스키마 | 손으로 JSON 작성 | 타입 힌트·독스트링에서 자동 생성 |
| 함수와 스키마 동기화 | **직접 챙겨야 함** | 자동 |
| 도구 실행 루프 | 직접 구현 (약 15줄) | SDK가 처리 |
| 여러 번 도구 호출 | 직접 반복문 작성 | SDK가 처리 |
| 추가 설치 | 없음 (`openai`만) | `openai-agents` 필요 |
| 다른 LLM으로 교체 | 같은 방식으로 가능 | SDK 지원 범위 안에서 |
| 내부 동작 이해 | **잘 보임** | 감춰짐 |

**어느 쪽을 쓰면 되나**

- **배울 때는 방법 A.** 도구 호출이 실제로 어떻게 오가는지 봐야 문제가 생겼을 때 고칠 수 있습니다.
- **만들 때는 방법 B.** 도구가 늘어날수록 손으로 쓴 스키마는 관리하기 어렵습니다.

방법 A를 먼저 해 본 지금은, 방법 B가 **무엇을 대신 해 주고 있는지** 알고 쓰는 셈입니다.
`openai_agent_code.ipynb`에서 이어서 다룹니다.

## 정리

| 개념 | 설명 |
|---|---|
| 도구 호출 | LLM이 "이 함수를 불러 달라"고 **요청**하는 것. 실행은 우리 코드가 한다 |
| JSON 스키마 | LLM에게 주는 함수 설명서. `description`이 호출 판단의 근거 |
| `finish_reason` | `"tool_calls"`면 도구 호출 요청, 아니면 보통 답변 |
| 2단계 호출 | 도구 실행 결과를 붙여 **다시 물어봐야** 최종 답변이 나온다 |
| `@function_tool` | 타입 힌트·독스트링에서 스키마를 자동 생성. 실행 루프도 SDK가 처리 |

**기억할 것**

- LLM은 함수를 실행하지 못합니다. **무엇을 부를지만** 정합니다.
- 언제 부를지는 **시스템 프롬프트**로, 어떻게 부를지는 **JSON 스키마**로 알려 줍니다.
- 도구가 실제로 불렸는지 확인하려면 함수 안에 `print`를 넣어 두세요.

## 스스로 해 보기

1. `description`을 `"도구"`처럼 성의 없이 바꾸고 다시 대화해 보세요. 모델이 제대로 호출하나요?
2. 시스템 프롬프트에서 "다 모이기 전에는 호출하지 말라"는 문장을 지워 보세요. 무엇이 달라지나요?
3. 상담 내용을 요약해 저장하는 두 번째 도구를 만들어 `tools` 목록에 추가해 보세요.
4. 지금 코드는 도구 호출을 **한 번만** 처리합니다. 2차 응답에서 모델이 또 도구를 부르면 어떻게 될까요?
   `while` 문으로 바꿔 여러 번 처리하도록 고쳐 보세요.

# 스스로 해 보기 — 풀이

Gradio로 손으로 대화하면 결과를 비교하기 어렵습니다. 그래서 **정해진 대화를 자동으로 흘려 보내는 도우미**를 먼저 만들고,
조건만 바꿔 가며 같은 대화를 여러 번 돌려 봅니다.

실험 1·2에서는 파일을 더럽히지 않도록 **기록하는 척만 하는 가짜 함수**를 씁니다.

## 준비: 실험용 도우미

`run_turns`는 사용자 발화를 차례로 보내고, 모델이 도구를 부르면 실행해 결과를 돌려줍니다.
도구 호출이 이어지는 동안 **`while` 문으로 계속 처리**하는데, 이것이 4번 문제의 답이기도 합니다(아래 4번에서 자세히 봅니다).

In [6]:
def run_turns(system, tools, user_turns, functions, model="gpt-5.4-mini", verbose=True):
    """user_turns를 차례로 보내며 대화하고, 불린 도구를 [(몇 번째 발화, 도구 이름, 인자)]로 돌려준다."""
    messages = [{"role": "system", "content": system}]
    calls = []
    for turn, user in enumerate(user_turns, 1):
        messages.append({"role": "user", "content": user})
        if verbose:
            print(f"[{turn}] 사용자: {user}")
        rounds = 0
        while True:  # 모델이 도구를 그만 부를 때까지 반복
            rounds += 1
            response = openai.chat.completions.create(model=model, messages=messages, tools=tools)
            msg = response.choices[0].message
            messages.append(msg)
            if not msg.tool_calls:
                break
            for tool_call in msg.tool_calls:
                args = json.loads(tool_call.function.arguments)
                calls.append((turn, tool_call.function.name, args))
                if verbose:
                    print(f"    → 도구 호출: {tool_call.function.name}({args})")
                result = functions[tool_call.function.name](**args)
                messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
        if verbose:
            print(f"    모델: {msg.content}")
            if rounds > 1:
                print(f"    (이번 발화에 API 호출 {rounds}회)")
            print()
    return calls


def fake_record(name=None, phone=None, address=None):
    """기록하는 척만 한다. 실험 중 personal_info.txt가 쌓이지 않게."""
    return "personal info is recorded"


# 정보를 한 번에 하나씩 알려 주는 대화. 도구는 3번째 발화에서만 불려야 정상이다.
partial_then_full = [
    "요즘 회사 일 때문에 고민이 많아요. 제 이름은 김철수예요.",
    "전화번호는 010-1234-5678이에요.",
    "주소는 서울시 강남구예요.",
]


def experiment(label, system, tools, n=3):
    """같은 대화를 n번 돌려서 도구가 언제, 어떤 값으로 불렸는지 요약한다."""
    print(f"== {label} ==")
    for i in range(n):
        calls = run_turns(system, tools, partial_then_full, {"record_personal_info_tool": fake_record}, verbose=False)
        summary = [f"{turn}번째 발화 {args}" for turn, _, args in calls] or ["호출 없음"]
        print(f"  시도 {i + 1}: " + " / ".join(summary))
    print()

In [7]:
# 기준: 원래 설정으로 한 번 자세히 본다.
run_turns(system_prompt, tools, partial_then_full, {"record_personal_info_tool": fake_record})

[1] 사용자: 요즘 회사 일 때문에 고민이 많아요. 제 이름은 김철수예요.


    모델: 회사 일 때문에 고민이 많으시군요. 요즘 어떤 점이 가장 힘드신지 편하게 말씀해 주세요.

상담 기록을 위해 **전화번호**와 **주소**도 알려주실 수 있을까요?  
지금까지는 **이름: 김철수**로 확인했습니다.

[2] 사용자: 전화번호는 010-1234-5678이에요.


    모델: 감사합니다. 아직 **주소**가 필요합니다.  
상담 기록을 위해 **주소**도 알려주세요.

[3] 사용자: 주소는 서울시 강남구예요.


    → 도구 호출: record_personal_info_tool({'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'})


    모델: 기록해 두었습니다.  
회사 일 때문에 어떤 고민이 가장 크신지 말씀해 주시면 함께 정리해볼게요.
    (이번 발화에 API 호출 2회)



[(3,
  'record_personal_info_tool',
  {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'})]

## 1. `description`을 성의 없이 바꾸면?

도구 설명을 `"도구"` 한 단어로 바꿉니다. 인자 설명도 함께 지워서 **스키마만으로는 무엇을 하는 도구인지 알 수 없게** 만듭니다.

그런데 지금 시스템 프롬프트에는 `record_personal_info_tool 도구를 호출해서 기록하세요`라고 **도구 이름이 직접 적혀 있습니다.**
모델이 설명 대신 프롬프트를 보고 판단할 수 있으므로, 도구 이름을 뺀 프롬프트(1-B)로도 시험합니다.

In [8]:
lazy_tool_json = {
    "name": "record_personal_info_tool",
    "description": "도구",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "phone": {"type": "string"},
            "address": {"type": "string"},
        },
        "required": ["name", "phone", "address"],
        "additionalProperties": False,
    },
}
lazy_tools = [{"type": "function", "function": lazy_tool_json}]

# 도구 이름을 말하지 않는 프롬프트
system_prompt_no_name = """
당신은 대화 전문가이며 대화하는 사람의 고민을 들어주고 제안을 할 수 있습니다.

대화 중에 상담 기록을 남기기 위해 사용자의 이름, 전화번호, 주소를 물어보세요.
세 가지를 모두 받으면 기록하세요.
세 가지가 다 모이기 전에는 기록하지 말고, 빠진 항목을 물어보세요.
"""

experiment("1-A. 성의 없는 설명 + 도구 이름이 적힌 프롬프트", system_prompt, lazy_tools)
experiment("1-B. 성의 없는 설명 + 도구 이름 없는 프롬프트", system_prompt_no_name, lazy_tools)
experiment("1-C. 제대로 된 설명 + 도구 이름 없는 프롬프트", system_prompt_no_name, tools)

== 1-A. 성의 없는 설명 + 도구 이름이 적힌 프롬프트 ==


  시도 1: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}


  시도 2: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}


  시도 3: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}

== 1-B. 성의 없는 설명 + 도구 이름 없는 프롬프트 ==


  시도 1: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}


  시도 2: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}


  시도 3: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}

== 1-C. 제대로 된 설명 + 도구 이름 없는 프롬프트 ==


  시도 1: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}


  시도 2: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}


  시도 3: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}



**결과와 해석** (gpt-5.4-mini, 각 3회)

- 1-A, 1-B, 1-C **모두 9번 중 9번** 3번째 발화에서 올바른 값으로 호출했습니다. 이번 실험에서는 차이가 나지 않았습니다.
- 이유는 **도구 이름**입니다. `record_personal_info_tool`과 인자 이름 `name`/`phone`/`address`만 봐도 무엇을 하는 도구인지 충분히 알 수 있습니다.
  설명이 없어도 이름이 설명 역할을 대신한 것입니다.
- `description`이 진짜 중요해지는 때는 **이름이 모호하거나**(`tool_1`, `save`) **비슷한 도구가 여러 개**일 때입니다.
  더 해 보기: 도구 이름을 `tool_1`로 바꾸고(1-B의 프롬프트에서) 같은 실험을 돌려 보세요.

## 2. "다 모이기 전에는 호출하지 말라"를 지우면?

In [9]:
system_prompt_no_rule = """
당신은 대화 전문가이며 대화하는 사람의 고민을 들어주고 제안을 할 수 있습니다.

대화 중에 상담 기록을 남기기 위해 사용자의 이름, 전화번호, 주소를 물어보세요.
세 가지를 모두 받으면 record_personal_info_tool 도구를 호출해서 기록하세요.
"""

experiment("2-A. 규칙 있음 (원래 프롬프트)", system_prompt, tools)
experiment("2-B. 규칙 없음", system_prompt_no_rule, tools)

== 2-A. 규칙 있음 (원래 프롬프트) ==


  시도 1: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}


  시도 2: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}


  시도 3: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}

== 2-B. 규칙 없음 ==


  시도 1: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}


  시도 2: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}


  시도 3: 3번째 발화 {'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'}



**결과와 해석** (gpt-5.4-mini, 각 3회)

- 규칙이 있을 때와 없을 때 **모두 3번째 발화에서만**, 지어낸 값 없이 호출했습니다. 이 모델에서는 차이가 없었습니다.
- 남은 문장 "세 가지를 모두 받으면 호출하라"와 스키마의 `required`만으로도 이 모델은 충분히 기다립니다.
  작은 모델이나 예전 모델에서는 빈 값·`"unknown"`으로 일찍 부르는 일이 실제로 자주 있었습니다.
- 그래서 규칙은 **모델이 바뀌어도 동작이 흔들리지 않게 하는 보험**으로 적어 둡니다.
- 프롬프트에만 기대지 않으려면 **함수 안에서 빈 값을 검사해 오류 메시지를 돌려주는** 방법이 더 확실합니다.
  모델은 그 메시지를 읽고 빠진 항목을 다시 물어봅니다.

## 3. 상담 요약을 저장하는 두 번째 도구

도구가 두 개가 되면 Step 6의 `chat()`처럼 함수를 하나로 고정해 부를 수 없습니다.
**도구 이름 → 파이썬 함수** 사전을 만들어 이름으로 찾아 부릅니다(`run_turns`의 `functions` 인자).

In [10]:
def save_consultation_summary_tool(name, summary):
    print(f"    [요약 저장] {name}: {summary}")
    with open("consultation_summary.txt", "a", encoding="utf-8") as f:
        f.write(f"{name}\t{summary}\n")
    return "consultation summary is saved"


save_consultation_summary_tool_json = {
    "name": "save_consultation_summary_tool",
    "description": "Use this tool to save a short summary of the counseling session when the user ends the conversation",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "The name of this user"},
            "summary": {"type": "string", "description": "One or two sentences: the user's concern and the advice given"},
        },
        "required": ["name", "summary"],
        "additionalProperties": False,
    },
}

tools3 = [
    {"type": "function", "function": record_personal_info_tool_json},
    {"type": "function", "function": save_consultation_summary_tool_json},
]

functions3 = {
    "record_personal_info_tool": fake_record,
    "save_consultation_summary_tool": save_consultation_summary_tool,
}

system_prompt3 = system_prompt + """
사용자가 상담을 마치려고 하면(고맙다고 하거나 끝내자고 하면) save_consultation_summary_tool 도구로
고민과 제안을 한두 문장으로 요약해서 저장하세요.
"""

calls = run_turns(system_prompt3, tools3, [
    "요즘 회사 일이 너무 많아서 잠을 잘 못 자요. 제 이름은 김철수예요.",
    "전화번호는 010-1234-5678, 주소는 서울시 강남구예요.",
    "퇴근하고 나서도 일 생각이 계속 나요. 어떻게 하면 좋을까요?",
    "좋은 방법이네요. 고마워요, 오늘은 여기까지 할게요.",
], functions3)

print("불린 도구:", [(turn, name) for turn, name, _ in calls])

[1] 사용자: 요즘 회사 일이 너무 많아서 잠을 잘 못 자요. 제 이름은 김철수예요.


    모델: 많이 지치셨겠어요. 일이 많아지면 몸도 마음도 계속 긴장해서 잠이 더 안 올 수 있어요.  
상담 기록을 위해 전화번호와 주소도 알려주실 수 있을까요?
많이 지치셨겠어요. 일이 많아지면 몸도 마음도 계속 긴장해서 잠이 더 안 올 수 있어요.  
상담 기록을 위해 전화번호와 주소도 알려주실 수 있을까요?

[2] 사용자: 전화번호는 010-1234-5678, 주소는 서울시 강남구예요.


    → 도구 호출: record_personal_info_tool({'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구'})


    모델: 기록해두었습니다.  
일이 너무 많아 잠을 못 주무시는 상황이라면, 오늘부터는 퇴근 후 30분 정도는 업무 생각을 끊고 가벼운 스트레칭이나 따뜻한 샤워로 몸을 먼저 이완해보세요. 잠들기 전에는 휴대폰과 업무 알림을 잠시 꺼두는 것도 도움이 됩니다.
    (이번 발화에 API 호출 2회)

[3] 사용자: 퇴근하고 나서도 일 생각이 계속 나요. 어떻게 하면 좋을까요?


    모델: 그럴 수 있어요. 업무량이 많을수록 머리가 계속 “아직 안 끝났다”고 느껴서 퇴근 후에도 생각이 이어지거든요.

도움이 되는 방법 몇 가지를 드리면:
- **퇴근 전에 5분 정리**: 오늘 할 일, 내일 할 일을 메모해 두면 뇌가 “기억해야 한다”는 부담을 조금 내려놓습니다.
- **퇴근 의식 만들기**: 회사 문을 나설 때 음악 듣기, 산책하기 같은 루틴을 정해두면 마음의 전환에 도움이 돼요.
- **생각 시간을 따로 정하기**: 집에 와서 바로 고민하지 말고, 저녁 중 10분만 “걱정 정리 시간”으로 잡아보세요.
- **몸부터 쉬게 하기**: 생각을 멈추려 애쓰기보다 샤워, 스트레칭, 호흡처럼 몸을 먼저 안정시키면 생각도 조금 느려집니다.
- **잠자기 전 업무 차단**: 메신저, 이메일 알림은 최소한 잠들기 1시간 전부터 끄는 걸 추천해요.

원하시면 제가 **퇴근 후 생각이 많을 때 바로 써먹는 5분 루틴**으로 더 구체적으로 정리해드릴게요.

[4] 사용자: 좋은 방법이네요. 고마워요, 오늘은 여기까지 할게요.


    → 도구 호출: save_consultation_summary_tool({'name': '김철수', 'summary': '회사 업무가 너무 많아 퇴근 후에도 생각이 멈추지 않고 잠을 잘 못 잔다고 하셨습니다. 퇴근 전 할 일 정리, 퇴근 의식 만들기, 생각 시간을 따로 두기, 잠들기 전 업무 알림 차단과 같은 방법을 제안했습니다.'})
    [요약 저장] 김철수: 회사 업무가 너무 많아 퇴근 후에도 생각이 멈추지 않고 잠을 잘 못 잔다고 하셨습니다. 퇴근 전 할 일 정리, 퇴근 의식 만들기, 생각 시간을 따로 두기, 잠들기 전 업무 알림 차단과 같은 방법을 제안했습니다.


    모델: 도움이 되셨다니 다행이에요. 오늘은 푹 쉬시고, 너무 무리하지 마세요.  
필요하실 때 또 편하게 찾아와 주세요.
    (이번 발화에 API 호출 2회)

불린 도구: [(2, 'record_personal_info_tool'), (4, 'save_consultation_summary_tool')]


In [11]:
print(open("consultation_summary.txt", encoding="utf-8").read())

김철수	회사 업무가 너무 많아 퇴근 후에도 생각이 멈추지 않고 잠을 잘 못 잔다고 하셨습니다. 퇴근 전 할 일 정리, 퇴근 의식 만들기, 생각 시간을 따로 두기, 잠들기 전 업무 알림 차단과 같은 방법을 제안했습니다.



## 4. 도구 호출을 여러 번 처리하기 (`while` 문)

Step 6의 `chat()`에는 문제가 두 가지 있습니다.

1. `if`라서 도구 호출을 **한 번만** 처리합니다. 2차 응답에서 모델이 또 도구를 부르면 그 요청은 처리되지 않고,
   `message.content`가 `None`인 채로 돌아가 **빈 답변**이 됩니다.
2. 도구 이름을 보지 않고 무조건 `record_personal_info_tool`을 부릅니다. 도구가 두 개면 **엉뚱한 함수가 실행**됩니다.

고친 방법은 `run_turns` 안에 이미 들어 있습니다.

```python
while True:
    response = 모델 호출
    if not msg.tool_calls:   # 도구를 안 부르면 최종 답변 → 끝
        break
    for tool_call in msg.tool_calls:
        functions[tool_call.function.name](**args)   # 이름으로 찾아서 실행
        결과를 tool 메시지로 추가
```

한 번에 모든 정보를 주고 바로 끝내자고 하면, 모델은 **기록 → 요약** 두 도구를 연달아(또는 한 번에 같이) 부릅니다.

In [12]:
calls = run_turns(system_prompt3, tools3, [
    "김철수, 010-1234-5678, 서울시 강남구입니다. 회사 일이 많아 잠을 못 자는데 "
    "퇴근 후 휴대폰 업무 알림을 끄라는 조언을 들었어요. 이걸로 상담 마칠게요, 고마워요."
], functions3)

[1] 사용자: 김철수, 010-1234-5678, 서울시 강남구입니다. 회사 일이 많아 잠을 못 자는데 퇴근 후 휴대폰 업무 알림을 끄라는 조언을 들었어요. 이걸로 상담 마칠게요, 고마워요.


    → 도구 호출: record_personal_info_tool({'name': '김철수', 'phone': '010-1234-5678', 'address': '서울시 강남구입니다.'})
    → 도구 호출: save_consultation_summary_tool({'name': '김철수', 'summary': '회사 일이 많아 잠을 못 자는 고민이 있었고, 퇴근 후 휴대폰 업무 알림을 끄는 방법을 조언했다.'})
    [요약 저장] 김철수: 회사 일이 많아 잠을 못 자는 고민이 있었고, 퇴근 후 휴대폰 업무 알림을 끄는 방법을 조언했다.


    모델: 상담을 마무리해주셔서 감사합니다. 편안한 휴식이 조금이라도 회복에 도움이 되길 바랍니다.
    (이번 발화에 API 호출 2회)



**결과 해석**

모델은 두 도구를 **한 응답 안에서 동시에**(`tool_calls`에 2개) 불렀습니다. 그래서 이번에는 API 호출이 2회로 끝났습니다.
그래도 Step 6의 `chat()`이었다면 문제가 생깁니다.

- `for` 문이 두 호출을 모두 `record_personal_info_tool`로 실행해서, 요약 호출 때 `phone`·`address`가 `None`인 **엉터리 줄이 파일에 기록**됩니다.
- 모델이 한 번에 하나씩 나눠 부르는 경우(자주 그렇습니다) 두 번째 요청은 **처리되지 않고 빈 답변**이 됩니다. `while` 문은 두 경우를 모두 처리합니다.

참고로 주소가 `서울시 강남구입니다.`로 저장되었습니다. 모델이 문장을 그대로 잘라 넣은 것인데,
인자 설명에 `"address only, without sentence endings"`처럼 **형식을 적어 주면** 줄어듭니다.

같은 방식으로 Gradio용 `chat()`을 고치면 이렇습니다. `functions3` 사전 덕분에 도구를 더 추가해도 이 함수는 고칠 필요가 없습니다.

In [13]:
def chat_v2(message, history):
    messages = [{"role": "system", "content": system_prompt3}]
    messages += [{"role": m["role"], "content": m["content"]} for m in history
                 if m.get("role") in ("user", "assistant") and isinstance(m.get("content"), str)]
    messages.append({"role": "user", "content": message})

    while True:
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools3)
        msg = response.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        for tool_call in msg.tool_calls:
            args = json.loads(tool_call.function.arguments)
            result = functions3[tool_call.function.name](**args)
            messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})


# 실제 파일에 기록하려면 fake_record 대신 진짜 함수를 연결한다.
functions3["record_personal_info_tool"] = record_personal_info_tool

# 직접 대화해 보려면 주석을 푸세요.
# gr.ChatInterface(chat_v2).launch(inbrowser=True)